In [1]:
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, roc_auc_score

print("--- SENTRi-X: Transfer Learning (Domain Adaptation) ---")

processed_dir = "../data/processed/"
models_dir = "../models/"

# 1. Load the locally scaled BoT-IoT data
X_bot = joblib.load(os.path.join(processed_dir, "bot_iot_X_test.pkl"))
y_bot = joblib.load(os.path.join(processed_dir, "bot_iot_y_test.pkl"))

# 2. Split: 20% for "Studying" (Fine-Tuning), 80% for the Final Exam
X_study, X_exam, y_study, y_exam = train_test_split(X_bot, y_bot, test_size=0.80, random_state=42, stratify=y_bot)

print(f"Data reserved for SENTRi-X to study: {X_study.shape[0]:,} packets")
print(f"Data reserved for the Final Exam: {X_exam.shape[0]:,} packets")

--- SENTRi-X: Transfer Learning (Domain Adaptation) ---
Data reserved for SENTRi-X to study: 28,945 packets
Data reserved for the Final Exam: 115,783 packets


In [2]:
print("Initiating Transfer Learning on Random Forest...")

# Train a new RF using the 20% study data
rf_adapted = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_adapted.fit(X_study, y_study)

# Test it on the 80% unseen data
adapted_preds = rf_adapted.predict(X_exam)
rf_probs_exam = rf_adapted.predict_proba(X_exam)[:, 1]

print("\n--- SENTRi-X RF Performance Post-Transfer Learning ---")
print(f"RF Adapted Accuracy: {accuracy_score(y_exam, adapted_preds):.4%}")
print("\nRF Detailed Report:")
print(classification_report(y_exam, adapted_preds, target_names=["Normal", "Attack"]))
print("\nRF Confusion Matrix:")
print(confusion_matrix(y_exam, adapted_preds))
print("\nRF ROC-AUC:", f"{roc_auc_score(y_exam, rf_probs_exam):.4f}")

Initiating Transfer Learning on Random Forest...

--- SENTRi-X RF Performance Post-Transfer Learning ---
RF Adapted Accuracy: 99.9870%

RF Detailed Report:
              precision    recall  f1-score   support

      Normal       0.88      0.33      0.48        21
      Attack       1.00      1.00      1.00    115762

    accuracy                           1.00    115783
   macro avg       0.94      0.67      0.74    115783
weighted avg       1.00      1.00      1.00    115783


RF Confusion Matrix:
[[     7     14]
 [     1 115761]]

RF ROC-AUC: 0.9761


In [3]:
# Save the fine-tuned BoT-IoT model to disk
save_path_bot = os.path.join(models_dir, "rf_model_bot_iot_finetuned.joblib")
joblib.dump(rf_adapted, save_path_bot)
print(f"Success: Adapted model saved to {save_path_bot}")

Success: Adapted model saved to ../models/rf_model_bot_iot_finetuned.joblib


In [4]:
# TRANSFER LEARNING FOR CNN & SAVING BOTH MODELS ---
from tensorflow.keras.models import load_model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np

print("Initiating Transfer Learning on CNN...")

# 1. Load the baseline ToN-IoT CNN model WITHOUT its old optimizer state
cnn_adapted = load_model(os.path.join(models_dir, "cnn_model_ton_iot.h5"), compile=False)

# 2. Recompile with a fresh optimizer for fine-tuning
cnn_adapted.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# 3. Reshape tabular data for the CNN (Requires a 3D tensor: samples, features, channels)
# Note: If your X_study is a Pandas DataFrame, we convert it to numpy using .values
features_count = X_study.shape[1]
X_study_cnn = np.reshape(
    X_study.values if hasattr(X_study, "values") else X_study,
    (X_study.shape[0], features_count, 1)
 )

# Also prepare the exam split for CNN evaluation
X_exam_cnn = np.reshape(
    X_exam.values if hasattr(X_exam, "values") else X_exam,
    (X_exam.shape[0], features_count, 1)
 )

# 4. Fine-tune the CNN on the 20% BoT-IoT study data
# We use early stopping to adapt without overfitting the new domain
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True,
    verbose=1
)

cnn_adapted.fit(
    X_study_cnn, 
    y_study, 
    epochs=20, 
    batch_size=32, 
    validation_split=0.1, 
    callbacks=[early_stopping],
    verbose=1
)

# 5. Evaluate the adapted CNN on the 80% exam data
cnn_probs_exam = cnn_adapted.predict(X_exam_cnn).ravel()
cnn_preds_exam = (cnn_probs_exam >= 0.5).astype(int)

print("\n--- SENTRi-X CNN Performance Post-Transfer Learning ---")
print(f"CNN Adapted Accuracy: {accuracy_score(y_exam, cnn_preds_exam):.4%}")
print("\nCNN Detailed Report:")
print(classification_report(y_exam, cnn_preds_exam, target_names=["Normal", "Attack"]))
print("\nCNN Confusion Matrix:")
print(confusion_matrix(y_exam, cnn_preds_exam))
print("\nCNN ROC-AUC:", f"{roc_auc_score(y_exam, cnn_probs_exam):.4f}")

# 6. Hybrid Soft-Voting Ensemble (RF + CNN) on exam split
ensemble_probs_exam = (rf_probs_exam + cnn_probs_exam) / 2.0
ensemble_preds_exam = (ensemble_probs_exam >= 0.5).astype(int)

print("\n--- SENTRi-X Hybrid Ensemble Performance (BoT-IoT) ---")
print(f"Ensemble Adapted Accuracy: {accuracy_score(y_exam, ensemble_preds_exam):.4%}")
print("\nEnsemble Detailed Report:")
print(classification_report(y_exam, ensemble_preds_exam, target_names=["Normal", "Attack"]))
print("\nEnsemble Confusion Matrix:")
print(confusion_matrix(y_exam, ensemble_preds_exam))
print("\nEnsemble ROC-AUC:", f"{roc_auc_score(y_exam, ensemble_probs_exam):.4f}")

# 7. SAVE BOTH ADAPTED MODELS TO DISK
print("\nSaving fine-tuned models to disk...")

rf_save_path = os.path.join(models_dir, "rf_model_bot_iot_finetuned.joblib")
joblib.dump(rf_adapted, rf_save_path)
print(f"Random Forest successfully saved to: {rf_save_path}")

cnn_save_path = os.path.join(models_dir, "cnn_model_bot_iot_finetuned.h5")
cnn_adapted.save(cnn_save_path)
print(f"CNN successfully saved to: {cnn_save_path}")

Initiating Transfer Learning on CNN...
Epoch 1/20
815/815 ━━━━━━━━━━━━━━━━━━━━ 11s 10ms/step - accuracy: 0.7314 - loss: 2.2039 - val_accuracy: 0.9955 - val_loss: 0.0455
Epoch 2/20
815/815 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.9994 - loss: 0.0883 - val_accuracy: 1.0000 - val_loss: 0.0019
Epoch 3/20
815/815 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.9997 - loss: 0.0665 - val_accuracy: 1.0000 - val_loss: 5.0634e-04
Epoch 4/20
815/815 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.9998 - loss: 0.0805 - val_accuracy: 1.0000 - val_loss: 3.0934e-04
Epoch 5/20
815/815 ━━━━━━━━━━━━━━━━━━━━ 10s 8ms/step - accuracy: 0.9998 - loss: 0.0639 - val_accuracy: 1.0000 - val_loss: 2.0892e-04
Epoch 6/20
815/815 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.9998 - loss: 0.0382 - val_accuracy: 1.0000 - val_loss: 1.6157e-04
Epoch 7/20
815/815 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.9998 - loss: 0.0625 - val_accuracy: 1.0000 - val_loss: 1.7342e-04
Epoch 8/20
815/815 ━━━━━━━━━━━━━━━━━━━━ 6s

[[     1     20]
 [     3 115759]]

Ensemble ROC-AUC: 0.9588

Saving fine-tuned models to disk...
Random Forest successfully saved to: ../models/rf_model_bot_iot_finetuned.joblib
CNN successfully saved to: ../models/cnn_model_bot_iot_finetuned.h5
